In [1]:
import pipelines
import pandas as pd
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('hotel_bookings.csv')

preparation = pipelines.default_data_preparation_pipeline
preparation.fit(df, None)
tmp_df = preparation.transform(df)


In [2]:
from ml_utils import run_experiment
from sklearn.pipeline import Pipeline
log_reg_pipe = Pipeline(
    [
        ('Data Preparation', pipelines.default_data_preparation_pipeline),
        ('model', LogisticRegression())
    ]
)
log_reg_pipe

Pipeline(steps=[('Data Preparation',
                 Pipeline(steps=[('Feature Engineering', FeatureEngineer()),
                                 ('feature encoding',
                                  ColumnTransformer(transformers=[('categorial_oh',
                                                                   OneHotEncoder(handle_unknown='ignore'),
                                                                   ['hotel',
                                                                    'arrival_date_month',
                                                                    'meal',
                                                                    'market_segment',
                                                                    'distribution_channel',
                                                                    'reserved_room_type',
                                                                    'deposit_type',
                                                                    'customer_type',
                                                                    'country_group']),
                                                                  ('nu...
                                                                    'adults',
                                                                    'children',
                                                                    'babies',
                                                                    'is_repeated_guest',
                                                                    'previous_cancellations',
                                                                    'previous_bookings_not_canceled',
                                                                    'booking_changes',
                                                                    'days_in_waiting_list',
                                                                    'required_car_parking_spaces',
                                                                    'total_of_special_requests',
                                                                    'total_nights',
                                                                    'is_weekend_stay',
                                                                    'total_guests',
                                                                    'has_children',
                                                                    'booking_intensity',
                                                                    'has_agent',
                                                                    'has_company'])]))])),
                ('model', LogisticRegression())])

### Тюн LogReg с помощью оптюны

In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 2.7 MB/s eta 0:00:00


In [4]:
log_reg_pipe.get_params()

{'memory': None,
 'steps': [('Data Preparation',
   Pipeline(steps=[('Feature Engineering', FeatureEngineer()),
                   ('feature encoding',
                    ColumnTransformer(transformers=[('categorial_oh',
                                                     OneHotEncoder(handle_unknown='ignore'),
                                                     ['hotel',
                                                      'arrival_date_month', 'meal',
                                                      'market_segment',
                                                      'distribution_channel',
                                                      'reserved_room_type',
                                                      'deposit_type',
                                                      'customer_type',
                                                      'country_group']),
                                                    ('numeric',
                                  

In [5]:
df = df.drop_duplicates()

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
import numpy as np
X = df.drop(columns='is_canceled')
y = df.is_canceled
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=22)
def objective(trial):
    max_iter = trial.suggest_int('model__max_iter', 1000, 10000, step=100)
    c = trial.suggest_float('model__C', 0.1, 10, log=True)
    solver = trial.suggest_categorical('model__solver', ['lbfgs', 'newton-cholesky'])
    if solver == 'lbfgs' or solver=='newton-cholesky':
        penalty = 'l2'
    else:
        penalty = trial.suggest_categorical('model__penalty', [None, 'l1', 'l2'])
    log_reg_pipe.set_params(
        model__max_iter=max_iter,
        model__solver=solver,
        model__penalty=penalty,
        model__C=c,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
    scores = cross_val_score(
        log_reg_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',
        n_jobs=-1,
        error_score='raise'
    )
    return np.mean(scores)

In [11]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=50)


[I 2026-06-16 16:40:54,958] A new study created in memory with name: no-name-d246e117-6a78-4f57-8a8b-cf1bc7f043e8


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-16 16:41:01,081] Trial 0 finished with value: 0.5351660088032698 and parameters: {'model__max_iter': 1700, 'model__C': 0.138523269590247, 'model__solver': 'newton-cholesky'}. Best is trial 0 with value: 0.5351660088032698.
[I 2026-06-16 16:41:06,322] Trial 1 finished with value: 0.5355245663916206 and parameters: {'model__max_iter': 8600, 'model__C': 0.2158715370708652, 'model__solver': 'lbfgs'}. Best is trial 1 with value: 0.5355245663916206.
[I 2026-06-16 16:41:14,023] Trial 2 finished with value: 0.5364761846337377 and parameters: {'model__max_iter': 5100, 'model__C': 7.145512213337412, 'model__solver': 'lbfgs'}. Best is trial 2 with value: 0.5364761846337377.
[I 2026-06-16 16:41:18,607] Trial 3 finished with value: 0.5353266784131521 and parameters: {'model__max_iter': 4200, 'model__C': 0.2523957960215724, 'model__solver': 'lbfgs'}. Best is trial 2 with value: 0.5364761846337377.
[I 2026-06-16 16:41:23,724] Trial 4 finished with value: 0.5353848630118022 and parameters: 

In [12]:
study.trials_dataframe().to_csv('logreg_results.csv')

### Тюн RandomForest

In [19]:
from sklearn.ensemble import RandomForestClassifier
rf_pipe = Pipeline(
    [('preparation', pipelines.default_data_preparation_pipeline),
     ('model', RandomForestClassifier())]
)
def objective_rf(trial):
    # Гиперпараметры RandomForest
    n_estimators = trial.suggest_int('model__n_estimators', 50, 500, step=60)
    max_depth = trial.suggest_int('model__max_depth', 3, 20, step=5)
    min_samples_split = trial.suggest_int('model__min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('model__min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('model__max_features', ['sqrt', 'log2', None])
    class_weight = trial.suggest_categorical('model__class_weight', ['balanced', None])
    bootstrap = trial.suggest_categorical('model__bootstrap', [True, False])

    # Устанавливаем параметры в пайплайн
    rf_pipe.set_params(
        model__n_estimators=n_estimators,
        model__max_depth=max_depth,
        model__min_samples_split=min_samples_split,
        model__min_samples_leaf=min_samples_leaf,
        model__max_features=max_features,
        model__class_weight=class_weight,
        model__bootstrap=bootstrap
    )

    # Кросс-валидация
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=22)
    scores = cross_val_score(
        rf_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',          # метрика для классификации
        n_jobs=-1,
        error_score='raise'
    )
    return np.mean(scores)

In [23]:

study = optuna.create_study(direction='maximize')
study.optimize(objective_rf, show_progress_bar=True, n_trials=20, n_jobs=-1)

[I 2026-06-16 17:15:30,434] A new study created in memory with name: no-name-120c94a5-f60b-4025-b3e7-a2d664d45ee0


  0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_382/3880085897.py:8: UserWarning: The distribution is specified by [50, 500] and step=60, but the range is not divisible by `step`. It will be replaced with [50, 470].
  n_estimators = trial.suggest_int('model__n_estimators', 50, 500, step=60)
/tmp/ipykernel_382/3880085897.py:9: UserWarning: The distribution is specified by [3, 20] and step=5, but the range is not divisible by `step`. It will be replaced with [3, 18].
  max_depth = trial.suggest_int('model__max_depth', 3, 20, step=5)       # можно заменить на suggest_categorical с None


[I 2026-06-16 17:19:14,929] Trial 0 finished with value: 0.6538164952685924 and parameters: {'model__n_estimators': 170, 'model__max_depth': 13, 'model__min_samples_split': 11, 'model__min_samples_leaf': 14, 'model__max_features': None, 'model__class_weight': None, 'model__bootstrap': True}. Best is trial 0 with value: 0.6538164952685924.


/tmp/ipykernel_382/3880085897.py:8: UserWarning: The distribution is specified by [50, 500] and step=60, but the range is not divisible by `step`. It will be replaced with [50, 470].
  n_estimators = trial.suggest_int('model__n_estimators', 50, 500, step=60)
/tmp/ipykernel_382/3880085897.py:9: UserWarning: The distribution is specified by [3, 20] and step=5, but the range is not divisible by `step`. It will be replaced with [3, 18].
  max_depth = trial.suggest_int('model__max_depth', 3, 20, step=5)       # можно заменить на suggest_categorical с None


...

[I 2026-06-16 17:36:59,454] Trial 16 finished with value: 0.6557964366332254 and parameters: {'model__n_estimators': 410, 'model__max_depth': 13, 'model__min_samples_split': 17, 'model__min_samples_leaf': 17, 'model__max_features': 'sqrt', 'model__class_weight': 'balanced', 'model__bootstrap': False}. Best is trial 3 with value: 0.6781693050123018.


/tmp/ipykernel_382/3880085897.py:8: UserWarning: The distribution is specified by [50, 500] and step=60, but the range is not divisible by `step`. It will be replaced with [50, 470].
  n_estimators = trial.suggest_int('model__n_estimators', 50, 500, step=60)
/tmp/ipykernel_382/3880085897.py:9: UserWarning: The distribution is specified by [3, 20] and step=5, but the range is not divisible by `step`. It will be replaced with [3, 18].
  max_depth = trial.suggest_int('model__max_depth', 3, 20, step=5)       # можно заменить на suggest_categorical с None


[I 2026-06-16 17:37:51,325] Trial 17 finished with value: 0.6571812104438829 and parameters: {'model__n_estimators': 410, 'model__max_depth': 13, 'model__min_samples_split': 17, 'model__min_samples_leaf': 17, 'model__max_features': 'sqrt', 'model__class_weight': 'balanced', 'model__bootstrap': False}. Best is trial 3 with value: 0.6781693050123018.


/tmp/ipykernel_382/3880085897.py:8: UserWarning: The distribution is specified by [50, 500] and step=60, but the range is not divisible by `step`. It will be replaced with [50, 470].
  n_estimators = trial.suggest_int('model__n_estimators', 50, 500, step=60)
/tmp/ipykernel_382/3880085897.py:9: UserWarning: The distribution is specified by [3, 20] and step=5, but the range is not divisible by `step`. It will be replaced with [3, 18].
  max_depth = trial.suggest_int('model__max_depth', 3, 20, step=5)       # можно заменить на suggest_categorical с None


[I 2026-06-16 17:39:12,185] Trial 18 finished with value: 0.6778628317742671 and parameters: {'model__n_estimators': 350, 'model__max_depth': 18, 'model__min_samples_split': 15, 'model__min_samples_leaf': 17, 'model__max_features': 'sqrt', 'model__class_weight': 'balanced', 'model__bootstrap': False}. Best is trial 3 with value: 0.6781693050123018.
[I 2026-06-16 17:46:46,127] Trial 19 finished with value: 0.6530458560489236 and parameters: {'model__n_estimators': 350, 'model__max_depth': 18, 'model__min_samples_split': 14, 'model__min_samples_leaf': 11, 'model__max_features': None, 'model__class_weight': 'balanced', 'model__bootstrap': False}. Best is trial 3 with value: 0.6781693050123018.


In [24]:
study.trials_dataframe().to_csv('rf_results.csv')